# 🎙️ Live Spanish Transcripts with Speakers from a YouTube Stream

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/real_time_streaming/YouTube_Live_Spanish_Speakers.ipynb)

Follow a live Spanish news broadcast on YouTube and get a transcript that says **who** is speaking, a few seconds behind the broadcast. When it ends, export the recording as a normal VideoDB video.

You will:
1. Connect a YouTube live URL as an RTStream.
2. Start transcription with the `pro` model and speaker labels.
3. See how speaker labels look in the data, word by word.
4. Read the transcript live, line by line, with speaker labels.
5. Stop the stream and export the recording with its audio.

**Requirements**
- A VideoDB **Pro plan**. YouTube live sources are billed $2 per hour on top of stream compute. The Free plan gets a `pro_plan_required` error.
- `videodb` SDK **0.5.4** or newer, for `model` and `speaker_labels` on `start_transcript`.

---
## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q "videodb>=0.5.4"

---
## 🔑 Step 2: Connect to VideoDB

Enter your `VIDEO_DB_API_KEY` when asked. The input is masked.

In [ ]:
import os
from getpass import getpass

import videodb

if not os.environ.get("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Please enter your VideoDB API Key: ")

conn = videodb.connect()
coll = conn.get_collection()
print("Connected to VideoDB")

---
## 📡 Step 3: Connect the YouTube Live Stream

We use RTVE Noticias, a live news channel from Spain. Any YouTube **live** URL works: a `watch?v=` link, a `/live/` link or a channel's `/@handle/live` page.

`store=True` keeps the recording so it can be exported at the end.

In [ ]:
YOUTUBE_LIVE_URL = "https://www.youtube.com/@rtvenoticias/live"

try:
    stream = coll.connect_rtstream(
        name="RTVE Noticias live",
        url=YOUTUBE_LIVE_URL,
        media_types=["video", "audio"],
        store=True,
    )
except Exception as error:
    # On the Free plan: "YouTube links require the Pro plan. Upgrade to Pro to continue."
    raise SystemExit(f"Could not connect the stream: {error}")

print(stream.id)

#### Already connected the stream? Reconnect with its id instead.

In [ ]:
# stream = coll.get_rtstream("rts-...")

---
## 🗣️ Step 4: Start Transcription with Speaker Labels

- `language_code="es"` sets Spanish.
- `model_name="pro"` uses the more accurate speech model. The default `standard` model also labels speakers for Spanish, English, French, German, Portuguese and Italian, at lower accuracy.
- `speaker_labels=True` labels each transcript record, and each word, with a speaker (`A`, `B`, ...).
- `max_speakers` caps how many distinct speakers it separates (1–10). Set it to the number of people you expect: extra voices reuse existing labels.

In [ ]:
stream.start_transcript(language_code="es", model_name="pro", speaker_labels=True, max_speakers=5)
print("Transcription started")

---
## 📝 Step 5: Read the Transcript Live

The first lines appear once the stream's worker is connected, usually within a minute or two. A line arrives when a speaker pauses, a few seconds behind the broadcast.

A news channel sometimes carries live events in other languages. `language_code="es"` steers the model toward Spanish, and the `pro` model still transcribes the other languages it hears.


In [ ]:
import time
from datetime import datetime

LISTEN_MINUTES = 5


def show_live_transcript(stream, minutes):
    seen, since = set(), None
    deadline = time.time() + minutes * 60
    while time.time() < deadline:
        page = stream.get_transcript(page_size=200, since=since) or {}
        for record in page.get("transcription_records") or []:
            key = (record.get("start"), record.get("text"))
            if key in seen:
                continue
            seen.add(key)
            since = max(since or 0, record.get("start") or 0)
            clock = datetime.fromtimestamp(record["start"]).strftime("%H:%M:%S") if record.get("start") else "--:--:--"
            print(f"[{clock}] Speaker {record.get('speaker') or '?'}: {record.get('text', '')}")
        time.sleep(5)
    print(f"\n{len(seen)} transcript lines")


show_live_transcript(stream, LISTEN_MINUTES)

---
## 🧩 Step 6: See the Speaker Data

A transcript record ends at a pause, not when the speaker changes, so one record can hold two speakers. Its `speaker` is the main one. Every word in `word_timestamps` carries its own `speaker`.

The cell below shows one raw record, then splits every record wherever the word speaker changes, and totals talk time per speaker.

In [ ]:
import json
from collections import defaultdict
from datetime import datetime


def all_records(stream):
    records, page = [], 1
    while True:
        data = stream.get_transcript(page=page, page_size=200) or {}
        records += data.get("transcription_records") or []
        if not data.get("next_page"):
            return records
        page += 1


def speaker_segments(record):
    """Split a record into runs of words with the same speaker."""
    segments = []
    for word in record.get("word_timestamps") or []:
        speaker = word.get("speaker") or record.get("speaker") or "?"
        if segments and segments[-1]["speaker"] == speaker:
            segments[-1]["words"].append(word["text"])
            segments[-1]["end"] = word["end"]
        else:
            segments.append({"speaker": speaker, "start": word["start"], "end": word["end"], "words": [word["text"]]})
    return segments


records = all_records(stream)

# 1. One raw record, as the API returns it (first 3 words shown)
sample = next((r for r in records if len({w.get("speaker") for w in r.get("word_timestamps") or []}) > 1), records[0])
print(json.dumps({**sample, "word_timestamps": sample["word_timestamps"][:3] + ["..."]}, ensure_ascii=False, indent=2))

# 2. Records split by speaker
print(f"\n{len(records)} records")
talk = defaultdict(float)
for number, record in enumerate(records, 1):
    segments = speaker_segments(record)
    mixed = " (mixed: " + ", ".join(s["speaker"] for s in segments) + ")" if len(segments) > 1 else ""
    print(f"\nRecord {number}, speaker {record.get('speaker')}{mixed}")
    for s in segments:
        talk[s["speaker"]] += s["end"] - s["start"]
        clock = datetime.fromtimestamp(s["start"]).strftime("%H:%M:%S")
        print(f"  [{clock}] Speaker {s['speaker']}: {' '.join(s['words'])}")

# 3. Talk time per speaker
print("\nTalk time")
for speaker, seconds in sorted(talk.items()):
    print(f"  Speaker {speaker}: {seconds:6.1f} s")

---
## ⏹️ Step 7: Stop the Stream

Stopping also stops transcription and billing. Always stop a stream you no longer need.

In [ ]:
stream.stop()
print("Stream stopped")

---
## 🎬 Step 8: Export the Recording

Export turns the stopped stream's recording into a regular VideoDB video, with its audio. It runs in the background: call it again until it returns the video.

In [ ]:
result = None
for _ in range(60):
    try:
        result = stream.export(name="RTVE Noticias live")
        if getattr(result, "video_id", None):
            break
    except Exception as error:
        print("Waiting for the recording:", error)
    time.sleep(10)

video = coll.get_video(result.video_id)
print("Exported:", video.id)
print(video.player_url)

---
## ✅ What You Built

- A live Spanish transcript, labelled by speaker, from a YouTube broadcast.
- A recording of the broadcast in your collection, ready to index, search and clip like any other video.

Next, index the exported video or attach live alerts to the stream. See the other notebooks in `real_time_streaming/`.